## Concept 4 — Hybrid RAG + Agentic RAG

### Part A: Hybrid RAG
Combines two retrieval methods:
- **Vector Search** — finds semantically similar docs (understands meaning)
- **BM25 Keyword Search** — finds exact keyword matches (like a search engine)

Each covers the other's blind spot. Together they find more relevant docs than either alone.

### Part B: Agentic RAG
Adds a self-correction loop:
1. Generate initial answer
2. LLM checks: is the answer sufficient?
3. If NO → refine the query and retrieve again
4. Return improved answer

### Limitation (what Concept 5 fixes)
Only one query variation is tried. If the user's phrasing is off, even a refined query may miss relevant docs.
→ Concept 5 (RAG Fusion) generates multiple query variations upfront.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries
from langchain_community.retrievers import BM25Retriever

chunks, vectorstore, embeddings, llm, prompt = setup()

### Part A — Hybrid RAG
#### Step 1: Create Both Retrievers

In [ ]:
# Vector retriever — semantic similarity
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# BM25 retriever — keyword matching (built from the same chunks)
bm25_retriever   = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

print("Both retrievers ready")

#### Step 2: Hybrid Retriever
Run both retrievers and merge results, removing duplicates.

In [ ]:
def hybrid_retrieve(query):
    docs_vector  = vector_retriever.invoke(query)
    docs_keyword = bm25_retriever.invoke(query)

    all_docs  = docs_vector + docs_keyword
    # Deduplicate by content
    unique_docs = list({doc.page_content: doc for doc in all_docs}.values())

    return unique_docs

# Test
query  = TEST_QUERIES[0]
result = hybrid_retrieve(query)
print(f"Query: {query}")
print(f"Vector: 3 docs | BM25: 3 docs | Merged unique: {len(result)} docs")

#### Step 3: Hybrid RAG Function

In [ ]:
def hybrid_rag(query):
    docs     = hybrid_retrieve(query)
    context  = format_docs(docs)
    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

run_queries(hybrid_rag, label="Concept 4A — Hybrid RAG")

### Part B — Agentic RAG
Adds a quality check loop on top of Hybrid RAG.

In [ ]:
def agentic_rag(query):
    print(f"\nQuery: {query}")

    # Step 1: initial retrieval + answer
    docs    = hybrid_retrieve(query)
    context = format_docs(docs)
    answer  = llm.invoke(prompt.format_messages(context=context, question=query)).content
    print(f"Initial docs: {len(docs)}")

    # Step 2: quality check
    check_prompt = f"Question: {query}\nAnswer: {answer}\n\nIs this answer sufficient? YES or NO only."
    check = llm.invoke(check_prompt).content.strip().upper()
    print(f"Quality check: {check}")

    # Step 3: refine if needed
    if "NO" in check:
        print("Refining query...")
        refine_prompt = (
            f"Improve this query for better search results (same intent):\n"
            f"Original: {query}\n\nReturn ONLY the improved query."
        )
        new_query = llm.invoke(refine_prompt).content.strip()
        print(f"Refined query: {new_query}")

        docs_refined  = hybrid_retrieve(new_query)
        docs_original = hybrid_retrieve(query)
        combined_docs = list({doc.page_content: doc for doc in (docs_refined + docs_original)}.values())
        print(f"Combined docs: {len(combined_docs)}")

        context = format_docs(combined_docs)
        answer  = llm.invoke(prompt.format_messages(context=context, question=query)).content

    return answer

### Test — Standard Queries

In [ ]:
run_queries(agentic_rag, label="Concept 4B — Agentic RAG")